# MedTube Segmentation — RGBD Training (4-channel) on Colab GPU

Trains YOLO11n-seg on 4-channel RGB-D images (RGB + depth grayscale as 4th channel).

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or A100)
2. Upload `medtube_rgbd.zip` to Google Drive root
3. Keep the Colab tab open during training (prevents idle timeout)

**Session protection:**
- `patience=50` prevents premature early stopping
- `save_period=10` saves checkpoint every 10 epochs to Drive
- If disconnected, use Cell 11 to resume from the last checkpoint

**Run all cells top to bottom — no jumping needed.**

In [ ]:
# Cell 1 — Install
!pip install ultralytics -q

In [ ]:
# Cell 2 — Check GPU
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4 — Unzip dataset from Google Drive
# 1. Upload medtube_rgbd.zip to your Google Drive root
# 2. Then run this cell to unzip it
import os

ZIP_PATH = '/content/drive/MyDrive/medtube_rgbd.zip'
if os.path.exists(ZIP_PATH):
    !unzip -q {ZIP_PATH} -d /content/
    print(f"Images: {len(os.listdir('/content/rgbd/images'))}")
    print(f"Labels: {len(os.listdir('/content/rgbd/labels'))}")
else:
    print(f"Upload medtube_rgbd.zip to Google Drive root first!")
    print(f"Expected at: {ZIP_PATH}")

In [ ]:
# Cell 5 — Keep-alive (prevents Colab idle timeout)
import threading, time

def keep_alive():
    while True:
        time.sleep(60)

thread = threading.Thread(target=keep_alive, daemon=True)
thread.start()
print("Keep-alive active — session will not idle-timeout")

In [ ]:
# Cell 6 — Create data.yaml for 4-channel RGBD
import os, glob

# Auto-detect where the images landed after unzip
candidates = glob.glob('/content/**/rgbd/images', recursive=True)
if not candidates:
    candidates = glob.glob('/content/**/images', recursive=True)
RGBD_ROOT = os.path.dirname(candidates[0]) if candidates else '/content/rgbd'
print(f"Dataset root: {RGBD_ROOT}")

data_yaml = f"""train: {RGBD_ROOT}/images
val: {RGBD_ROOT}/images
test: {RGBD_ROOT}/images

nc: 4
names: ['Other', 'Push-on', 'Screwcap', 'Universal']
"""

with open(f'{RGBD_ROOT}/data.yaml', 'w') as f:
    f.write(data_yaml)

print('data.yaml created')
print(f"Images: {len(os.listdir(f'{RGBD_ROOT}/images'))}")
print(f"Labels: {len(os.listdir(f'{RGBD_ROOT}/labels'))}")

In [ ]:
# Cell 7 — Patch YOLO model for 4-channel input (ch=4)
# YOLO expects 3-channel RGB by default. We modify the first conv layer
# to accept 4 channels, keeping pretrained weights for RGB and adding
# a zero-initialized 4th channel.

from ultralytics import YOLO
import torch
import copy

model = YOLO('yolo11n-seg.pt')  # Using YOLO11n as base (smaller, faster)

# Get the first conv layer
first_conv = model.model.model[0].conv
old_weight = first_conv.weight.data  # shape: [out_ch, 3, kH, kW]

# Create new conv with 4 input channels
new_conv = torch.nn.Conv2d(
    4, old_weight.shape[0],
    kernel_size=first_conv.kernel_size,
    stride=first_conv.stride,
    padding=first_conv.padding,
    bias=first_conv.bias is not None,
)

# Copy RGB weights, zero-init the depth channel
with torch.no_grad():
    new_conv.weight[:, :3, :, :] = old_weight
    new_conv.weight[:, 3:, :, :] = 0.0  # depth channel starts at zero
    if first_conv.bias is not None:
        new_conv.bias = copy.deepcopy(first_conv.bias)

# Replace in model
model.model.model[0].conv = new_conv
model.model.model[0].conv.in_channels = 4

# Update model yaml to reflect 4 channels
if hasattr(model.model, 'yaml'):
    model.model.yaml['ch'] = 4

print(f'First conv patched: {first_conv.in_channels}ch → 4ch')
print(f'Weight shape: {new_conv.weight.shape}')

In [ ]:
# Cell 8 — Train (with session-timeout protections)
DRIVE_SAVE = '/content/drive/MyDrive/medtube_runs'

results = model.train(
    data     = f'{RGBD_ROOT}/data.yaml',
    epochs   = 100,
    imgsz    = 640,
    batch    = 16,
    patience = 50,       # increased from 20 — prevents premature early stopping
    save_period = 10,    # save checkpoint every 10 epochs (resume if disconnected)
    workers  = 2,
    device   = 0,
    project  = DRIVE_SAVE,
    name     = 'YOLO11n-RGBD',
    exist_ok = True,
    # Mild augmentation (no colour jitter — would distort depth channel)
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=20.0, translate=0.05, scale=0.25,
    shear=2.0, perspective=0.0001,
    flipud=0.0, fliplr=0.5,
    mosaic=0.2, erasing=0.2,
    auto_augment='randaugment',
)

print(f'\nBest weights: {DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')

In [ ]:
# Cell 9 — Evaluate
best = YOLO(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')
r = best.val(data=f'{RGBD_ROOT}/data.yaml', split='test', imgsz=640, device=0)
print(f'Box  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}')
print(f'Mask mAP50={r.seg.map50:.4f}  mAP50-95={r.seg.map:.4f}')

In [ ]:
# Cell 10 — Download
from google.colab import files
files.download(f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')

In [ ]:
# Cell 11 — Resume after disconnect (run Cells 1-7 first, then this instead of Cell 8)
# This resumes training from the last saved checkpoint on Drive.
DRIVE_SAVE = '/content/drive/MyDrive/medtube_runs'
LAST_PT = f'{DRIVE_SAVE}/YOLO11n-RGBD/weights/last.pt'

import os
if os.path.exists(LAST_PT):
    from ultralytics import YOLO
    model = YOLO(LAST_PT)
    model.train(resume=True)
    print(f'Resumed and completed. Best: {DRIVE_SAVE}/YOLO11n-RGBD/weights/best.pt')
else:
    print(f'No checkpoint found at {LAST_PT} — run Cell 8 for fresh training')